In [1]:
!pip install transformers==4.30.2
!pip install timm==0.6.12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 12.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 75.1 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 13.3 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.20
    Uninstalling timm-1.0.20:
      Successf

In [2]:
import torch
import json
import os
from PIL import Image

class ENTMedClipDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, metadata_path):
        self.img_dir = img_dir
        self.items = []
        with open(metadata_path) as f:
            for line in f:
                self.items.append(json.loads(line))

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path = os.path.join(self.img_dir, "images", item["image"])
        image = Image.open(img_path).convert("RGB")
        text  = item["text"]
        return {"image": image, "text": text}

    def __len__(self):
        return len(self.items)

In [3]:
import torch
import torch.nn as nn
from transformers import BlipProcessor, BlipModel

class BLIPForRetrieval(nn.Module):
    def __init__(self):
        super().__init__()
        # image encoder + text encoder
        self.model = BlipModel.from_pretrained("Salesforce/blip-itm-base-coco")

        # chiếu về cùng không gian (optional)
        self.proj = nn.Linear(self.model.config.hidden_size, 512)

    def forward(self, pixel_values=None, input_ids=None, attention_mask=None):
        outputs = self.model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        image_embeds = outputs.vision_model_output.pooler_output
        text_embeds  = outputs.text_model_output.pooler_output

        # chiếu cùng dimension
        image_embeds = self.proj(image_embeds)
        text_embeds  = self.proj(text_embeds)

        # normalize
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        text_embeds  = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        # similarity logits
        logits = image_embeds @ text_embeds.T

        return {
            "logits": logits,
            "img_embeds": image_embeds,
            "text_embeds": text_embeds,
        }

2026-01-04 01:25:51.901346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767489952.063257      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767489952.109133      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767489952.485833      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767489952.485872      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767489952.485874      23 computation_placer.cc:177] computation placer alr

In [4]:
from transformers import BlipProcessor

processor = BlipProcessor.from_pretrained("Salesforce/blip-itm-base-coco")

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"]  for x in batch]
    return {"image": images, "text": texts}


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [5]:
from torch.utils.data import DataLoader
import json

root = "/kaggle/input/sumotosima/sumotosima_split"

train_ds = ENTMedClipDataset(
    img_dir=f"{root}/train",
    metadata_path=f"{root}/train/metadata_train.jsonl"
)

val_ds = ENTMedClipDataset(
    img_dir=f"{root}/val",
    metadata_path=f"{root}/val/metadata_val.jsonl"
)

test_ds = ENTMedClipDataset(
    img_dir=f"{root}/test",
    metadata_path=f"{root}/test/metadata_test.jsonl"
)

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))
print("Test samples :", len(test_ds))

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2
)

print("✔ Loaded train/val/test DataLoader thành công!")


Train samples: 400
Val samples  : 50
Test samples : 50
✔ Loaded train/val/test DataLoader thành công!


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import BlipModel
from tqdm.auto import tqdm
import os

# Redefine the BLIPForRetrieval class with the fix for hidden_size
class BLIPForRetrieval(nn.Module):
    def __init__(self):
        super().__init__()
        # image encoder + text encoder
        self.model = BlipModel.from_pretrained("Salesforce/blip-itm-base-coco")

        # chiếu về cùng không gian (optional)
        # Corrected line: access hidden_size from the text_model's config
        self.proj = nn.Linear(self.model.text_model.config.hidden_size, 512)

    def forward(self, pixel_values=None, input_ids=None, attention_mask=None):
        outputs = self.model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        image_embeds = outputs.vision_model_output.pooler_output
        text_embeds  = outputs.text_model_output.pooler_output

        # chiếu cùng dimension
        image_embeds = self.proj(image_embeds)
        text_embeds  = self.proj(text_embeds)

        # normalize
        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        text_embeds  = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        # similarity logits
        logits = image_embeds @ text_embeds.T

        return {
            "logits": logits,
            "img_embeds": image_embeds,
            "text_embeds": text_embeds,
        }

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BLIPForRetrieval().to(device)
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-3)

best_val_loss = float("inf")
best_path = "/content/blip_best.pth"

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        inputs = processor(
            images=batch["image"],
            text=batch["text"],
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = model(
            pixel_values=inputs["pixel_values"],
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        logits = outputs["logits"]
        labels = torch.arange(logits.size(0)).to(device)

        # contrastive loss
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"[Epoch {epoch+1}] Loss = {total_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            inputs = processor(
                images=batch["image"],
                text=batch["text"],
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            outputs = model(
                pixel_values=inputs["pixel_values"],
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"]
            )

            logits = outputs["logits"]
            labels = torch.arange(logits.size(0)).to(device)

            loss = (
                F.cross_entropy(logits, labels) +
                F.cross_entropy(logits.T, labels)
            ) / 2

            val_loss += loss.item()

    val_loss /= len(val_loader)
    print("Val =", val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)
        print("🔥 Saved best model")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/895M [00:00<?, ?B/s]

`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.


model.safetensors:   0%|          | 0.00/895M [00:00<?, ?B/s]

Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-itm-base-coco and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_model.encoder.layer.0.attention.self.value.bias', 'text_model.encoder.layer.0.attention.self.value.weight', 'text_model.encoder.layer.0.crossattention.output.LayerNorm.bias', 'text_model.encoder.

...

  0%|          | 0/50 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[Epoch 10] Loss = 1.3923


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Val = 1.7228763444083077
🔥 Saved best model


In [7]:
import numpy as np

model = BLIPForRetrieval().to(device)
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

image_embeds = []
text_embeds = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        inputs = processor(
            images=batch["image"],
            text=batch["text"],
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = model(
            pixel_values=inputs["pixel_values"],
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        img = outputs["img_embeds"].cpu().numpy()
        txt = outputs["text_embeds"].cpu().numpy()

        image_embeds.append(img)
        text_embeds.append(txt)

image_embeds = np.vstack(image_embeds)
text_embeds = np.vstack(text_embeds)

`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-itm-base-coco and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_model.enco

  0%|          | 0/50 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [8]:
import numpy as np

# =========================================================
# Recall@K
# =========================================================
def recall_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    hits = np.array([1 if i in ranking[i, :k] else 0 for i in range(N)])

    return hits.mean()


# =========================================================
# Precision@K
# =========================================================
def precision_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    precisions = np.array([
        (1 if i in ranking[i, :k] else 0) / k
        for i in range(N)
    ])

    return precisions.mean()


# =========================================================
# Average Precision cho 1 sample
# =========================================================
def average_precision(ranking, gt_index):
    score = 0.0
    correct = 0
    for rank, idx in enumerate(ranking, start=1):
        if idx == gt_index:
            correct += 1
            score += correct / rank
    return score


# =========================================================
# Mean Average Precision (mAP)
# =========================================================
def mean_average_precision(sim_matrix):
    N = sim_matrix.shape[0]
    ranking = np.argsort(-sim_matrix, axis=1)

    aps = np.array([
        average_precision(ranking[i], i)
        for i in range(N)
    ])
    return aps.mean()


# =========================================================
# nDCG@K
# =========================================================
def ndcg_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    ndcgs = []

    for i in range(N):
        topk = ranking[i, :k]
        rel = np.array([1 if idx == i else 0 for idx in topk])

        dcg = np.sum(rel / np.log2(np.arange(2, k+2)))
        idcg = 1.0  # vì chỉ có 1 ground truth
        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)

In [9]:
sim_matrix = image_embeds @ text_embeds.T

print("Recall@1 :", recall_at_k(sim_matrix, 1))
print("Recall@5 :", recall_at_k(sim_matrix, 5))
print("Recall@10:", recall_at_k(sim_matrix, 10))

print("Precision@1:", precision_at_k(sim_matrix, 1))
print("Precision@5:", precision_at_k(sim_matrix, 5))
print("Precision@10:", precision_at_k(sim_matrix, 10))


print("mAP:", mean_average_precision(sim_matrix))

print("nDCG@1:", ndcg_at_k(sim_matrix, 1))
print("nDCG@5:", ndcg_at_k(sim_matrix, 5))
print("nDCG@10:", ndcg_at_k(sim_matrix, 10))

Recall@1 : 0.08
Recall@5 : 0.5
Recall@10: 1.0
Precision@1: 0.08
Precision@5: 0.10000000000000002
Precision@10: 0.09999999999999998
mAP: 0.2744920634920635
nDCG@1: 0.08
nDCG@5: 0.281717322994755
nDCG@10: 0.44021228282012104


In [10]:
import numpy as np

def compute_mrr(sim_matrix):
    N = sim_matrix.shape[0]
    rr = []
    for i in range(N):
        ranks = np.argsort(-sim_matrix[i])
        pos = np.where(ranks == i)[0][0]
        rr.append(1 / (pos + 1))
    return np.mean(rr)

print("MRR (text → image):", compute_mrr(sim_matrix))

MRR (text → image): 0.2744920634920635
